# Evaluation setup and smoke test

Fixes **one** classification prompt, **one** set of generation options and **one** category parser for all four candidate models, then checks them on **non-golden** team rows only.

The logic lives in the `.py` modules beside this notebook, so the notebook, the triage service and the accuracy run all use the same code:

| File | Contents |
| --- | --- |
| `prompt_template.md` | The single classification prompt (`{narrative}` is replaced by the ticket text) |
| `eval_config.json` | Model pins (tag → digest), labels, generation options, Ollama address |
| `classifier.py` | `build_prompt`, `parse_label`, `classify`, pin and CPU checks |
| `data.py` | Team rows, golden IDs, non-golden development rows, golden guard |
| `measure_num_ctx.py` | Ticket lengths and the `num_ctx` choice |
| `smoke_test.py` | CPU-only check and the smoke test |

No golden narrative is sent to any model here; `data.assert_not_golden` enforces this.

In [1]:
import importlib
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "classifier.py").is_file() else Path.cwd() / "evaluation"
sys.path.insert(0, str(HERE))

import classifier, data, measure_num_ctx, smoke_test
for module in (classifier, data, measure_num_ctx, smoke_test):
    importlib.reload(module)  # picks up edits to the prompt or config without restarting the kernel

## 1. Team rows, golden IDs and development rows

In [2]:
team = data.load_team_rows()
golden = data.load_golden()
dev_rows = data.load_dev_rows(team)
print(f"Team rows: {len(team)} | golden: {len(golden)} | development (non-golden): {len(dev_rows)}")

Team rows: 1000 | golden: 175 | development (non-golden): 825


## 2. Frozen configuration, parser check and model pins

The prompt (`prompt_template.md`) and settings (`eval_config.json`) are frozen; this section checks them and the model pins.

In [3]:
print(classifier.PROMPT_TEMPLATE)
print("\nOptions:", classifier.OPTIONS, "| keep_alive:", classifier.KEEP_ALIVE, "| Ollama:", classifier.OLLAMA_URL)
print("Parser rules:", *classifier.PARSER_RULES, sep="\n  ")

You are a complaint triage classifier for a financial services company.
Classify the customer complaint below into exactly one of these seven categories, based on the principal problem the customer is complaining about.

Categories:
- Credit reporting: accuracy, disclosure, dispute, or correction of a credit report.
- Debt collection: attempts to collect a claimed debt, including contact, validation, demands, threats, or collection conduct.
- Mortgage: home-loan origination, payments, servicing, escrow, modification, or foreclosure.
- Credit card: card-account billing, interest, fees, charges, or transaction disputes.
- Bank account or service: deposit-account access, deposits, balances, fees, or account servicing.
- Consumer loan: non-mortgage, non-card borrowing, including student, auto, and personal loans.
- Money transfer or service: sending, receiving, cancelling, or recovering a transfer, wire, remittance, or transfer-service transaction.

Rules:
- If the customer requests a clea

In [4]:
classifier.self_test()

Parser self-check passed (11 cases)
Ollama version: 0.35.0
llama3.2:1b   digest OK | 1.32 GB | 1.2B | Q8_0
qwen2.5:1.5b  digest OK | 0.99 GB | 1.5B | Q4_K_M
phi3.5:3.8b   digest OK | 2.18 GB | 3.8B | Q4_0
qwen2.5:7b    digest OK | 4.68 GB | 7.6B | Q4_K_M


## 3. Ticket lengths and `num_ctx`

Lengths use all 1,000 team rows; token counts use only the 20 longest development rows. If the recommendation differs from the current value, set `write=True` (or edit `eval_config.json`) and re-run from the top.

In [5]:
stats = measure_num_ctx.length_stats(team)
print("Narrative characters (team rows):", stats)

_, needed, recommended = measure_num_ctx.measure(team, dev_rows, list(classifier.MODELS))
print(f"\nRecommended num_ctx (10% margin): {recommended} (current: {classifier.OPTIONS['num_ctx']})")

write = False
if write:
    measure_num_ctx.write_num_ctx(recommended)
    print("Written to eval_config.json: re-run from the top")

Narrative characters (team rows): {'rows': 1000, 'min': 200, 'median': 787.0, 'p95': 1783, 'p99': 1931, 'max': 1997}
llama3.2:1b   template 302 tok | worst 0.234 tok/char | longest team ticket needs ~786 tok
qwen2.5:1.5b  template 306 tok | worst 0.242 tok/char | longest team ticket needs ~805 tok
phi3.5:3.8b   template 361 tok | worst 0.279 tok/char | longest team ticket needs ~935 tok
qwen2.5:7b    template 306 tok | worst 0.242 tok/char | longest team ticket needs ~805 tok

Recommended num_ctx (10% margin): 2048 (current: 4096)


## 4. CPU-only check

`size_vram` must be 0 for every candidate.

In [6]:
cpu_check = smoke_test.check_cpu_only(list(classifier.MODELS))

llama3.2:1b   size_vram 0 -> CPU only
qwen2.5:1.5b  size_vram 0 -> CPU only
phi3.5:3.8b   size_vram 0 -> CPU only
qwen2.5:7b    size_vram 0 -> CPU only


## 5. Smoke test on development rows

4 seeded development rows per **source** category (28 rows), each sent once to every candidate after a warm-up. Results are written to `smoke_results/`.

In [7]:
smoke_rows = smoke_test.sample_rows(dev_rows, per_source=4)
print(f"{len(smoke_rows)} smoke rows:", sorted(int(r["row"]) for r in smoke_rows))

results = smoke_test.run(list(classifier.MODELS), smoke_rows)
smoke_test.summarise(results)

28 smoke rows: [1008, 1076, 1120, 1136, 1191, 1196, 1212, 1253, 1305, 1341, 1345, 1370, 1490, 1501, 1528, 1529, 1535, 1549, 1594, 1617, 1660, 1715, 1837, 1903, 1910, 1930, 1950, 1967]
llama3.2:1b done -> C:\Users\izzul\Documents\Github\ict3113-p1-1-a1\evaluation\smoke_results\smoke_llama3.2_1b.csv
qwen2.5:1.5b done -> C:\Users\izzul\Documents\Github\ict3113-p1-1-a1\evaluation\smoke_results\smoke_qwen2.5_1.5b.csv
phi3.5:3.8b done -> C:\Users\izzul\Documents\Github\ict3113-p1-1-a1\evaluation\smoke_results\smoke_phi3.5_3.8b.csv
qwen2.5:7b done -> C:\Users\izzul\Documents\Github\ict3113-p1-1-a1\evaluation\smoke_results\smoke_qwen2.5_7b.csv

model           valid errors trunc  =source  median s  max s
llama3.2:1b    27/28       0     0    8/28       3.44   6.36
qwen2.5:1.5b   28/28       0     0   12/28       3.12   4.40
phi3.5:3.8b    28/28       0     0   18/28       6.25  14.35
qwen2.5:7b     28/28       0     0   20/28       9.62  20.42

Outputs the parser could not map (fix the parser 